In [23]:
#2021.08.18. WED
#Team_밥믈리에

#00. 패키지 호출
import pandas as pd 
import numpy as np 
import re 
import itertools
import csv
import konlpy
from konlpy.tag import Okt
from collections import Counter

#00-1. 사전변수 정의하기.
CULTIVAR = '고시히카리'

STOPWORD = []
with open('../data/stop_word_list.csv', 'r') as file :
    reader = csv.reader(file)
    for row in reader                                :
        STOPWORD.append(row)
STOPWORD = STOPWORD[0]


In [24]:
#01. 데이터셋 결합하기. 
#(1) 쿠팡 데이터셋 불러오기. 
review_data_01 = pd.read_excel(f'../data/Coupang/{CULTIVAR}.xlsx')

#(2) 쿠팡 데이터셋 확인하기.
review_data_01

,name,kg,delivery,rate,date,review,rice
0,이화에월백하고 고시히까리 3kg 강화섬쌀 ES강화농산 강화쌀 고시히까리 강화섬쌀 강...,3kg,무료배송,5,2021.06.06,NaN,NaN
1,대한농산 보약같은 고시히카리 쌀,5kg,무료배송,5,2021.08.14,NaN,NaN
2,2020년햅쌀/슈퍼오닝/특등급쌀/고시히카리/쌀20Kg/쌀10Kg/쌀4Kg/안중농협쌀...,4kg,무료배송,5,2021.07.21,NaN,NaN
3,2020년햅쌀/슈퍼오닝/특등급쌀/고시히카리/쌀20Kg/쌀10Kg/쌀4Kg/안중농협쌀...,5kg,무료배송,5,2021.06.24,NaN,표면 매끈해요\n도정일 최근이에요\n쌀알 보통이에요
4,2020년햅쌀/슈퍼오닝/특등급쌀/고시히카리/쌀20Kg/쌀10Kg/쌀4Kg/안중농협쌀...,6kg,무료배송,5,2021.05.16,NaN,NaN
...,...,...,...,...,...,...,...
13078,한가위쌀 김포 고시히카리쌀 백미,20kg,무료배송,4,2016.01.19,NaN,NaN
13079,한가위쌀 김포 고시히카리쌀 백미,20kg,무료배송,4,2016.01.13,밥이 맛있네요...윤기도 좔좔~~~두번째 구매합니다.번창하십시요,NaN
13080,한가위쌀 김포 고시히카리쌀 백미,20kg,무료배송,4,2015.12.19,빠른 배송 너무 좋아요,NaN
13081,한가위쌀 김포 고시히카리쌀 백미,20kg,무료배송,5,2015.11.17,맛있는 김포 고시히카리쌀 잘 받았습니다♥\n배송도 빠르고~아주 만족!,NaN


In [25]:
#(3) 롯데 데이터셋 불러오기.
review_data_02 = pd.read_excel('../data/crawling_LotteON.xlsx', sheet_name = CULTIVAR)

#(4) 롯데 데이터셋 확인하기. 
review_data_02 

,product_name,cultivar,review_01,review_02,score,date
0,엘그로 완전미 평택 고시히카리 (5KG),고시히카리,재구매,NO_VALUE,5,1시간 전
1,엘그로 완전미 평택 고시히카리 (5KG),고시히카리,온라인 구입시 품절이 잘되요,NO_VALUE,5,8시간 전
2,엘그로 완전미 평택 고시히카리 (5KG),고시히카리,고시하카리 소량판매 좋아요,NO_VALUE,5,2021.08.17
3,엘그로 완전미 평택 고시히카리 (5KG),고시히카리,잘 받았습니다~~~,NO_VALUE,5,2021.08.16
4,엘그로 완전미 평택 고시히카리 (5KG),고시히카리,한달사용기,NO_VALUE,5,2021.08.14
...,...,...,...,...,...,...
1318,[2020년산] 고시히카리 10kg,고시히카리,당일도정은 아니고 8월6일 도정된거네요. 기존에 먹던게 있는데 요즘 안나와서ㅜ일단 ...,“보통이에요”,5,2020.08.19
1319,윤기가 흐르는 고시히카리쌀 5kg 안성양성농협,고시히카리,고시히까리 인데 별차이모르겠어요\n그저그래요,“보통이에요”,4,2020.07.04
1320,[온세일]2020년 햅쌀 백미/고시히카리/추청쌀 10kg 모음전,고시히카리,마트판매가보다 저렴하게 구입할수 있어 우선 구매했습니다.\n아직 사용전이니 나중에 ...,“재구매의사 있어요”,4,2020.07.13
1321,[2020년산] 고시히카리 20kg,고시히카리,물품 잘 받았습니다. 감사합니다.^^,“재구매의사 있어요”,5,2020.06.15


In [26]:
#(5) 데이터셋 취합하기. 
review_data = pd.DataFrame(map(str, review_data_01['review'].append(review_data_02['review_01'])))

#(6) 취합 데이터셋 확인하기.
len(review_data), type(review_data)

(14406, pandas.core.frame.DataFrame)

In [27]:
#02. 데이터셋 전처리하기. 
#(1) 결측값 제거하기. 
review_data[review_data[0]=='nan'] = np.nan
review_data = review_data.dropna(axis=0)

#(3) 리스트로 변환하기. 
review_data = review_data[0].values.tolist()

#(4) 데이터셋 구조 확인하기. 
review_data

['맛있습니다\n맛있습니다 잘먹고있어요',
 '아주맛있다까진아니고 괜찮은편입니다',
 '밥맛이 진짜 조아요~\n밥맛이 진짜 조으네요~',
 '배송 겁내 빠름',
 '밥맛이 정말 좋아요.\n아쉬운 것은 혈당 관리 때문에 잡곡밥을 먹어야 해서...',
 '좋아요 ㅎㅎ\n잘 먹을께요',
 '밥할 때 냄새가 나고 밥맛이 텁텁하여 썩 좋은 느낌은 아니었습니다',
 '바구미 한마리가 기어다니는걸 보고 실망',
 '도정일이 6월이더라고요ㅜㅠ\n쌀이 하얗지 않고 냄새도 좀 나요ㅠㅠ',
 '아이 이유식하려고 샀어요~\n쌀 상태 좋습니다!',
 '굿굿\n항상 맛잇게 먹고있어요',
 '평택슈퍼오닝알아주는쌀이라주문함',
 '밥냄새도 너무 좋고 맛있어요',
 '쌀 괜찮아요 밥맛도 맛있어요!',
 '걱정안해도 되네요 좋아요\n품질좋아요.',
 '맛있음\n믿고먹는 평택쌀',
 '맛있어요. 재구매의사 있습니다.',
 '물건이 안왔는데 왜 배송완료죠??',
 '전 이쌀구매해먹은지 2년 넘엇어요 밥맛도 최고 윤기도짱',
 '재구매 하려고요~',
 '난 고시히카리만 먹는데 어느 순간 매장에 산거랑 인터넷에 산거랑 다른다는 생각이 많이듬.\n인터넷에 산 고시히카리 제대로 된것 못봤슴.\n이것도 별로네요.밥 맛이.',
 '최애 쌀입니다',
 '가장맛있는쌀',
 '진짜맛있는 쌀입니다',
 '맛있어요',
 '맛 있네요.. 담에 또 구매 할께요',
 '햅쌀이라는데 재고 느낌',
 '저는 작년부터 이 쌀만 먹는데 쿠팡에 있길래 쿠팡을 통해서도 두번째 구매입니다\n주문받고 도정해주셔서 기간은 4-5일 걸리는데 그래도 쌀이 너무 좋아요\n깨끗하고 밥도 참 이쁘게 고슬고슬 됩니다 다른 쌀 못 먹겠더라구요\n배송기간이 있어서 잠시 4키로짜리 좋다는 걸로 사서 먹으면서 기다렸는데 별로.. 이*쌀 강*섬쌀 등등 다 먹어봤는데 이게 최고에요~\n그래서 떨어지기 며칠전 미리 주문해놓습니당',
 '평택살았을때 맛이 생각나서 시켰는데 최악이네요ㅠㅠ',
 '맛있어요',
 '찰지고 맛있어요 만족합니다 ^^~~',
 '맛나요~

In [28]:
#03. 텍스트 전처리하기. 
#(1) 텍스트에서 한글, 영어, 띄어쓰기를 제외하고 모두 삭제하기. 
review_data_pp  = [] 
for i in range(len(review_data)) :
    text = re.sub('[^ㄱ-ㅎ가-힣a-zA-Z0-9 ]', '', review_data[i])
    review_data_pp.append(text)
    
#(2) 처리 결과 확인하기. 
review_data_pp

['맛있습니다맛있습니다 잘먹고있어요',
 '아주맛있다까진아니고 괜찮은편입니다',
 '밥맛이 진짜 조아요밥맛이 진짜 조으네요',
 '배송 겁내 빠름',
 '밥맛이 정말 좋아요아쉬운 것은 혈당 관리 때문에 잡곡밥을 먹어야 해서',
 '좋아요 ㅎㅎ잘 먹을께요',
 '밥할 때 냄새가 나고 밥맛이 텁텁하여 썩 좋은 느낌은 아니었습니다',
 '바구미 한마리가 기어다니는걸 보고 실망',
 '도정일이 6월이더라고요쌀이 하얗지 않고 냄새도 좀 나요',
 '아이 이유식하려고 샀어요쌀 상태 좋습니다',
 '굿굿항상 맛잇게 먹고있어요',
 '평택슈퍼오닝알아주는쌀이라주문함',
 '밥냄새도 너무 좋고 맛있어요',
 '쌀 괜찮아요 밥맛도 맛있어요',
 '걱정안해도 되네요 좋아요품질좋아요',
 '맛있음믿고먹는 평택쌀',
 '맛있어요 재구매의사 있습니다',
 '물건이 안왔는데 왜 배송완료죠',
 '전 이쌀구매해먹은지 2년 넘엇어요 밥맛도 최고 윤기도짱',
 '재구매 하려고요',
 '난 고시히카리만 먹는데 어느 순간 매장에 산거랑 인터넷에 산거랑 다른다는 생각이 많이듬인터넷에 산 고시히카리 제대로 된것 못봤슴이것도 별로네요밥 맛이',
 '최애 쌀입니다',
 '가장맛있는쌀',
 '진짜맛있는 쌀입니다',
 '맛있어요',
 '맛 있네요 담에 또 구매 할께요',
 '햅쌀이라는데 재고 느낌',
 '저는 작년부터 이 쌀만 먹는데 쿠팡에 있길래 쿠팡을 통해서도 두번째 구매입니다주문받고 도정해주셔서 기간은 45일 걸리는데 그래도 쌀이 너무 좋아요깨끗하고 밥도 참 이쁘게 고슬고슬 됩니다 다른 쌀 못 먹겠더라구요배송기간이 있어서 잠시 4키로짜리 좋다는 걸로 사서 먹으면서 기다렸는데 별로 이쌀 강섬쌀 등등 다 먹어봤는데 이게 최고에요그래서 떨어지기 며칠전 미리 주문해놓습니당',
 '평택살았을때 맛이 생각나서 시켰는데 최악이네요',
 '맛있어요',
 '찰지고 맛있어요 만족합니다 ',
 '맛나요 완전 맛나요꾸준히 주문해서 먹었는데 한번은 가격이 너무 비싸져서 다른 쌀 먹어봤는데 슈퍼오닝이 제 입맛을 버릇없게 만들었

In [29]:
#04. 자연어처리해 형태소 분석하기.
#(1) okt(twitter) 객체 정의하기
okt = Okt()

#(2) 한 레코드 씩 형태소 분석하기.  
extract_value_list = []
for i in range (len(review_data)) :
    extract_value = okt.morphs(review_data[i], stem=True, norm=True)
    extract_value_list.append(extract_value)

#(3) 처리 결과 확인하기. 
extract_value_list

[['맛있다', '\n', '맛있다', '잘', '먹다'],
 ['아주', '맛있다', '아니다', '괜찮다', '편입', '니', '다'],
 ['밥맛', '이', '진짜', '좋다', '~', '\n', '밥맛', '이', '진짜', '좋다', '~'],
 ['배송', '겁내', '빠르다'],
 ['밥맛',
  '이',
  '정말',
  '좋다',
  '.',
  '\n',
  '아쉽다',
  '것',
  '은',
  '혈당',
  '관리',
  '때문',
  '에',
  '잡곡',
  '밥',
  '을',
  '먹다',
  '하다',
  '...'],
 ['좋다', 'ㅎㅎ', '\n', '자다', '먹다'],
 ['밥',
  '하다',
  '때',
  '냄새',
  '가',
  '나다',
  '밥맛',
  '이',
  '텁텁하',
  '여',
  '썩',
  '좋다',
  '느낌',
  '은',
  '아니다'],
 ['바구미', '한', '마리', '가', '기다', '보고', '실망'],
 ['도정',
  '일이',
  '6월',
  '이더라고요',
  'ㅜㅠ',
  '\n',
  '쌀',
  '이',
  '하얗다',
  '않다',
  '냄새',
  '도',
  '좀',
  '나요',
  'ㅠㅠ'],
 ['아이', '이유식', '하다', '사다', '~', '\n', '쌀', '상태', '좋다', '!'],
 ['굿굿', '\n', '항상', '맛', '잇다', '먹다'],
 ['평택', '슈퍼', '오', '닝', '알아주다', '쌀', '이라', '주문', '함'],
 ['밥', '냄새', '도', '너무', '좋다', '맛있다'],
 ['쌀', '괜찮다', '밥맛', '도', '맛있다', '!'],
 ['걱정', '안해', '도', '되다', '좋다', '\n', '품질', '좋다', '.'],
 ['맛있다', '\n', '믿다', '먹다', '평택', '쌀'],
 ['맛있다', '.', '재다', '의사', '있다', '.'],
 ['물건', '

In [30]:
#(3) 한 리스트로 저장하기.
morpheme_list = []
for i in range(len(extract_value_list)) :
    morpheme_list.extend(extract_value_list[i])

#(4) 처리 결과 확인하기. 
morpheme_list

['맛있다',
 '\n',
 '맛있다',
 '잘',
 '먹다',
 '아주',
 '맛있다',
 '아니다',
 '괜찮다',
 '편입',
 '니',
 '다',
 '밥맛',
 '이',
 '진짜',
 '좋다',
 '~',
 '\n',
 '밥맛',
 '이',
 '진짜',
 '좋다',
 '~',
 '배송',
 '겁내',
 '빠르다',
 '밥맛',
 '이',
 '정말',
 '좋다',
 '.',
 '\n',
 '아쉽다',
 '것',
 '은',
 '혈당',
 '관리',
 '때문',
 '에',
 '잡곡',
 '밥',
 '을',
 '먹다',
 '하다',
 '...',
 '좋다',
 'ㅎㅎ',
 '\n',
 '자다',
 '먹다',
 '밥',
 '하다',
 '때',
 '냄새',
 '가',
 '나다',
 '밥맛',
 '이',
 '텁텁하',
 '여',
 '썩',
 '좋다',
 '느낌',
 '은',
 '아니다',
 '바구미',
 '한',
 '마리',
 '가',
 '기다',
 '보고',
 '실망',
 '도정',
 '일이',
 '6월',
 '이더라고요',
 'ㅜㅠ',
 '\n',
 '쌀',
 '이',
 '하얗다',
 '않다',
 '냄새',
 '도',
 '좀',
 '나요',
 'ㅠㅠ',
 '아이',
 '이유식',
 '하다',
 '사다',
 '~',
 '\n',
 '쌀',
 '상태',
 '좋다',
 '!',
 '굿굿',
 '\n',
 '항상',
 '맛',
 '잇다',
 '먹다',
 '평택',
 '슈퍼',
 '오',
 '닝',
 '알아주다',
 '쌀',
 '이라',
 '주문',
 '함',
 '밥',
 '냄새',
 '도',
 '너무',
 '좋다',
 '맛있다',
 '쌀',
 '괜찮다',
 '밥맛',
 '도',
 '맛있다',
 '!',
 '걱정',
 '안해',
 '도',
 '되다',
 '좋다',
 '\n',
 '품질',
 '좋다',
 '.',
 '맛있다',
 '\n',
 '믿다',
 '먹다',
 '평택',
 '쌀',
 '맛있다',
 '.',
 '재다',
 '의사',
 '있다',
 '.',
 '물건',


In [31]:
#(5) 특수문자 공백처리하기. 
morpheme_list_pp  = [] 
for i in range(len(morpheme_list)) :
    text = re.sub('[^가-힣 ]', '', morpheme_list[i])
    morpheme_list_pp.append(text)

#(6) 처리 결과 확인하기. 
morpheme_list_pp

['맛있다',
 '',
 '맛있다',
 '잘',
 '먹다',
 '아주',
 '맛있다',
 '아니다',
 '괜찮다',
 '편입',
 '니',
 '다',
 '밥맛',
 '이',
 '진짜',
 '좋다',
 '',
 '',
 '밥맛',
 '이',
 '진짜',
 '좋다',
 '',
 '배송',
 '겁내',
 '빠르다',
 '밥맛',
 '이',
 '정말',
 '좋다',
 '',
 '',
 '아쉽다',
 '것',
 '은',
 '혈당',
 '관리',
 '때문',
 '에',
 '잡곡',
 '밥',
 '을',
 '먹다',
 '하다',
 '',
 '좋다',
 '',
 '',
 '자다',
 '먹다',
 '밥',
 '하다',
 '때',
 '냄새',
 '가',
 '나다',
 '밥맛',
 '이',
 '텁텁하',
 '여',
 '썩',
 '좋다',
 '느낌',
 '은',
 '아니다',
 '바구미',
 '한',
 '마리',
 '가',
 '기다',
 '보고',
 '실망',
 '도정',
 '일이',
 '월',
 '이더라고요',
 '',
 '',
 '쌀',
 '이',
 '하얗다',
 '않다',
 '냄새',
 '도',
 '좀',
 '나요',
 '',
 '아이',
 '이유식',
 '하다',
 '사다',
 '',
 '',
 '쌀',
 '상태',
 '좋다',
 '',
 '굿굿',
 '',
 '항상',
 '맛',
 '잇다',
 '먹다',
 '평택',
 '슈퍼',
 '오',
 '닝',
 '알아주다',
 '쌀',
 '이라',
 '주문',
 '함',
 '밥',
 '냄새',
 '도',
 '너무',
 '좋다',
 '맛있다',
 '쌀',
 '괜찮다',
 '밥맛',
 '도',
 '맛있다',
 '',
 '걱정',
 '안해',
 '도',
 '되다',
 '좋다',
 '',
 '품질',
 '좋다',
 '',
 '맛있다',
 '',
 '믿다',
 '먹다',
 '평택',
 '쌀',
 '맛있다',
 '',
 '재다',
 '의사',
 '있다',
 '',
 '물건',
 '이',
 '안',
 '오다',
 '왜',
 '배송',
 '완료'

In [32]:
#(7) 데이터 프레임 변환하기. 
morpheme_df = pd.DataFrame({'value': morpheme_list_pp})

#(8) 불용어 처리하기. 
for i in range(len(STOPWORD)) :
    morpheme_df[morpheme_df['value']==STOPWORD[i]] = np.nan

#(9) 데이터프레임 보기 좋게 구성하기.
word_frequency = morpheme_df.value_counts().to_frame()
word_frequency = word_frequency.reset_index()
word_frequency.columns = ['value','count']
word_frequency

#(10) 단어 글자 수가 1 이하인 값 처리하기
for i in range(len(word_frequency))            :
    try                                        :
        if len(word_frequency['value'][i]) < 2 :
            word_frequency.at[i ,'count'] = 0
    except Exception                           :
        print(f'ERROR ! index_num = {i}')

#(11) count 값이 5 이상인 행만 출력하기. 
word_frequency = word_frequency[word_frequency['count'] >= 5]

#(12) 엑셀로 저장하기.
word_frequency.to_excel(f'../data/NLP/{CULTIVAR}_word_frequency.xlsx', index=False)

#(13) 결과 확인하기. 
word_frequency

,value,count
10,윤기,919
27,곱다,341
37,찰지다,255
38,지다,251
75,찰기,111
83,고소하다,97
88,구수하다,88
96,돼다,79
99,촉촉하다,76
112,달다,61
